In [ ]:
import pandas as pd

df_train = pd.read_csv('../data/original/train.csv')
df_test = pd.read_csv('../data/original/test.csv')
df_complete = pd.read_csv('../data/original/titanic3.csv')

In [ ]:
# Find names in `df_test` that contain ", which is not properly formatted in the source.
df_test[df_test['Name'].str.contains("\"", na=False)]

In [ ]:
# Find names in df_complete that contain "
df_complete[df_complete['name'].str.contains("\"", na=False)]

In [ ]:
# Remove " from names containing " from both dataframes
df_test['Name'] = df_test['Name'].str.replace('"', '', regex=False)
df_complete['name'] = df_complete['name'].str.replace('"', '', regex=False)

In [ ]:
y_train = df_train['Survived']
X_train = df_train.drop('Survived', axis=1)

In [ ]:
X_test = df_test.copy()

In [ ]:
# Check records from df_complete where `name` is not unique
df_complete[df_complete.duplicated(subset=['name'], keep=False)]

In [ ]:
# Merge the column `Survived` from df_complete as integer into df_test based on the `name` and `age` column from `df_complete` and `Name` and `Age` columns `df_test` ordered by `PassengerId`
y_test = df_test.merge(df_complete[['name', 'age', 'survived']], left_on=['Name', 'Age'], right_on=['name', 'age'], how='left')['survived']

In [ ]:
# Display rows from `y_test` where `survived` is null
y_test

In [ ]:
from sap_rpt_oss import SAP_RPT_OSS_Classifier
from sklearn.metrics import accuracy_score

In [ ]:
# Initialize a classifier
clf = SAP_RPT_OSS_Classifier(max_context_size=8192, bagging=8)

In [ ]:
clf.fit(X_train, y_train)

In [ ]:
# Predict probabilities
prediction_probabilities = clf.predict_proba(X_test)
# Predict labels
predictions = clf.predict(X_test)

In [ ]:
print("Accuracy", accuracy_score(y_test, predictions))

In [ ]:
# Combine y_test, predictions into a dataframe
results_df = pd.DataFrame({'Actual': y_test, 'Predicted': predictions})

In [ ]:
results_df.head(10)

In [ ]:
grouped = results_df.groupby(['Actual', 'Predicted']).size()
print(grouped)